# Phase 5: Backtesting, Validation & Research Hygiene  
## Notebook 05_09 — White Reality Check Bootstrap

### Research question

How do we test whether the best strategy in a large strategy library truly outperforms a benchmark, after correcting for data snooping and multiple strategy trials using bootstrap methods?

This notebook follows:

```text
05_06_performance_haircut_and_deflated_sharpe
05_07_purged_kfold_and_embargo_cv
05_08_probability_of_backtest_overfitting
```

The previous notebook estimated the probability that a strategy-selection process overfits. This notebook asks a related but distinct question:

> After testing many strategies, is the best observed performance statistically significant, or could it be explained by data snooping?

It covers:

1. White's Reality Check motivation;
2. benchmark-relative performance series;
3. null hypothesis of no superior strategy;
4. data-snooping bias;
5. IID bootstrap;
6. moving block bootstrap;
7. stationary bootstrap;
8. max-performance test statistic;
9. centered bootstrap under the null;
10. White Reality Check p-value;
11. conservative nature of WRC;
12. simplified Hansen SPA-style test;
13. effect of poor strategies on Reality Check;
14. placebo all-noise library;
15. true-alpha library;
16. bootstrap confidence intervals;
17. sensitivity to block length;
18. sensitivity to performance measure;
19. reality-check adjusted strategy report;
20. governance flags;
21. saved outputs and manifest.

The key idea:

> White's Reality Check asks whether the best strategy remains impressive after comparing it to the best strategy that could have emerged from noise across the entire search.

## 1. Data-snooping problem

Suppose we evaluate $M$ strategies.

For strategy $m$, define benchmark-relative performance:

$$
d_{m,t} = r_{m,t} - r_{0,t}
$$

where $r_{0,t}$ is the benchmark return.

A naive test might check whether:

$$
\bar d_m > 0
$$

for the best strategy.

But if many strategies were tried, the maximum:

$$
\max_m \bar d_m
$$

is biased upward.

White's Reality Check tests whether the best observed outperformance is larger than what could occur from searching many strategies under the null that none has positive expected outperformance.

## 2. White Reality Check null

The null hypothesis is:

$$
H_0: \max_m E[d_{m,t}] \le 0
$$

Meaning:

> No strategy in the tested library has positive expected performance relative to the benchmark.

The alternative is:

$$
H_1: \exists m \text{ such that } E[d_{m,t}] > 0
$$

The test statistic is often:

$$
T = \max_m \sqrt{n}\bar d_m
$$

or a studentized version.

The bootstrap estimates the distribution of this maximum statistic under the null.

## 3. Centering under the null

To impose the null, bootstrap the centered performance differences:

$$
\tilde d_{m,t} = d_{m,t} - \bar d_m
$$

Then each strategy has zero mean in the bootstrap world.

For bootstrap sample $b$:

$$
T_b^* = \max_m \sqrt{n}\bar{\tilde d}_{m}^{*(b)}
$$

Reality Check p-value:

$$
p = P^*(T_b^* \ge T_{obs})
$$

A small p-value means the best strategy is unlikely to be explained by data snooping alone.

## 4. Why block bootstrap?

Financial returns are often serially dependent.

IID bootstrap breaks time dependence.

Block bootstrap preserves local dependence.

We compare:

1. IID bootstrap;
2. moving block bootstrap;
3. stationary bootstrap.

The stationary bootstrap randomly changes block length and is useful when dependence length is uncertain.

## 5. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class WhiteRealityCheckConfig:
    n_days: int = 1_800
    annualisation: int = 252
    seed: int = 42
    n_strategies: int = 300
    n_true_alpha: int = 12
    bootstrap_samples: int = 1_500
    block_length: int = 20
    stationary_bootstrap_p: float = 1 / 20
    cvar_alpha: float = 0.95
    transaction_cost_bps_low: float = 1.0
    transaction_cost_bps_high: float = 8.0
    significance_level: float = 0.05
    studentize_epsilon: float = 1e-8
    spa_exclusion_threshold: float = -1.0

config = WhiteRealityCheckConfig()
config

## 6. Simulate strategy library

We simulate a library of strategy variants and a benchmark.

Most strategies are noise or weakly correlated variants.

A small subset has weak positive true alpha.

This lets us compare Reality Check behaviour in:

1. mixed true-alpha library;
2. all-noise placebo library;
3. stronger-alpha library.

In [ ]:
def simulate_strategy_library(
    config: WhiteRealityCheckConfig,
    scenario_name: str,
    true_alpha_scale: float = 1.0,
    all_noise: bool = False,
):
    rng = np.random.default_rng(config.seed + abs(hash(scenario_name)) % 100_000)
    dates = pd.bdate_range("2016-01-01", periods=config.n_days)

    strategy_ids = [f"strategy_{i:03d}" for i in range(config.n_strategies)]

    regime = np.zeros(config.n_days, dtype=int)
    stress_type = np.array(["normal"] * config.n_days, dtype=object)

    market = rng.standard_t(df=6, size=config.n_days) * 0.009
    rates = rng.standard_t(df=6, size=config.n_days) * 0.003
    style = rng.standard_t(df=6, size=config.n_days) * 0.006

    for t in range(1, config.n_days):
        if regime[t - 1] == 0:
            regime[t] = rng.choice([0, 1], p=[0.985, 0.015])
        else:
            regime[t] = rng.choice([0, 1], p=[0.060, 0.940])

        if regime[t] == 1:
            market[t] *= 2.4
            rates[t] *= 2.0
            style[t] *= 1.8

        u = rng.uniform()
        if u < 0.010:
            stress_type[t] = "risk_off"
            market[t] -= rng.uniform(0.020, 0.080)
            rates[t] += rng.uniform(0.001, 0.010)
        elif u < 0.016:
            stress_type[t] = "risk_on_rebound"
            market[t] += rng.uniform(0.015, 0.050)

    benchmark = 0.65 * market + 0.25 * rates + rng.standard_t(df=8, size=config.n_days) * 0.004

    n = config.n_strategies
    true_alpha_daily = np.zeros(n)

    if not all_noise and config.n_true_alpha > 0:
        alpha_idx = rng.choice(n, size=config.n_true_alpha, replace=False)
        true_alpha_daily[alpha_idx] = rng.uniform(0.00005, 0.00022, size=config.n_true_alpha) * true_alpha_scale

    beta_market = rng.normal(0.15, 0.35, size=n)
    beta_rates = rng.normal(0.02, 0.20, size=n)
    beta_style = rng.normal(0.10, 0.25, size=n)

    idio_vol = rng.uniform(0.005, 0.018, size=n)
    tail_loading = rng.uniform(0.0, 0.014, size=n)
    turnover = rng.uniform(0.05, 0.90, size=n)
    cost_bps = rng.uniform(config.transaction_cost_bps_low, config.transaction_cost_bps_high, size=n)

    returns = np.zeros((config.n_days, n))

    for t in range(config.n_days):
        vol_mult = 1.0 if regime[t] == 0 else 2.2
        common = beta_market * market[t] + beta_rates * rates[t] + beta_style * style[t]

        idio = rng.standard_t(df=5, size=n) * np.sqrt((5 - 2) / 5) * idio_vol * vol_mult

        tail = np.zeros(n)
        if stress_type[t] == "risk_off":
            tail = -tail_loading * rng.uniform(0.3, 1.2, size=n)
        elif stress_type[t] == "risk_on_rebound":
            tail = 0.25 * tail_loading * rng.uniform(0.2, 0.8, size=n)

        cost_drag = turnover * cost_bps / 10000.0 / 10.0

        returns[t] = true_alpha_daily + common + idio + tail - cost_drag

    returns_df = pd.DataFrame(returns, index=dates, columns=strategy_ids)
    benchmark = pd.Series(benchmark, index=dates, name="benchmark_return")

    metadata = pd.DataFrame({
        "strategy_id": strategy_ids,
        "true_alpha_daily": true_alpha_daily,
        "true_alpha_ann": true_alpha_daily * config.annualisation,
        "has_true_alpha": true_alpha_daily > 0,
        "beta_market": beta_market,
        "beta_rates": beta_rates,
        "beta_style": beta_style,
        "idio_vol": idio_vol,
        "tail_loading": tail_loading,
        "turnover": turnover,
        "cost_bps": cost_bps,
        "scenario_name": scenario_name,
    })

    regime_info = pd.DataFrame({
        "market": market,
        "rates": rates,
        "style": style,
        "benchmark_return": benchmark,
        "regime": regime,
        "stress_type": stress_type,
    }, index=dates)

    return returns_df, benchmark, metadata, regime_info

strategy_returns, benchmark, strategy_metadata, regime_info = simulate_strategy_library(config, "mixed_true_alpha")

strategy_returns.head(), benchmark.head(), strategy_metadata.head()

In [ ]:
plt.figure(figsize=(12, 6))
for col in strategy_returns.columns[:10]:
    plt.plot(strategy_returns.index, (1 + strategy_returns[col]).cumprod(), alpha=0.65, label=col)
plt.plot(benchmark.index, (1 + benchmark).cumprod(), color="black", linewidth=2, label="benchmark")
plt.title("Example Strategy Equity Curves vs Benchmark")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend(ncol=3)
plt.show()

## 7. Benchmark-relative performance matrix

The Reality Check works with relative returns:

$$
d_{m,t} = r_{m,t} - r_{0,t}
$$

where $r_0$ is the benchmark.

In [ ]:
relative_returns = strategy_returns.sub(benchmark, axis=0)

relative_returns.head()

## 8. Performance metrics

White's Reality Check is commonly expressed in terms of mean performance differentials.

We also test variants based on:

- mean daily excess return;
- annualised Sharpe of excess returns;
- Sortino of excess returns;
- Calmar of excess returns.

The bootstrap test statistic can be changed, but the multiple-testing principle remains the same.

In [ ]:
def annualised_return(returns, annualisation=252):
    r = pd.Series(returns).dropna()
    if len(r) == 0:
        return np.nan
    return float((1 + r).prod() ** (annualisation / len(r)) - 1)

def annualised_vol(returns, annualisation=252):
    r = pd.Series(returns).dropna()
    if len(r) < 2:
        return np.nan
    return float(r.std(ddof=1) * np.sqrt(annualisation))

def sharpe_ratio(returns, annualisation=252):
    r = pd.Series(returns).dropna()
    sd = r.std(ddof=1)
    if len(r) < 2 or sd == 0:
        return np.nan
    return float(r.mean() / sd * np.sqrt(annualisation))

def sortino_ratio(returns, annualisation=252):
    r = pd.Series(returns).dropna()
    downside = r[r < 0]
    sd = downside.std(ddof=1)
    if len(downside) < 2 or sd == 0:
        return np.nan
    return float(r.mean() / sd * np.sqrt(annualisation))

def max_drawdown_from_returns(returns):
    r = pd.Series(returns).dropna()
    nav = (1 + r).cumprod()
    dd = nav / nav.cummax() - 1
    return float(dd.min()) if len(dd) else np.nan

def calmar_ratio(returns, annualisation=252):
    ann = annualised_return(returns, annualisation)
    mdd = max_drawdown_from_returns(returns)
    if not np.isfinite(mdd) or mdd >= 0:
        return np.nan
    return float(ann / abs(mdd))

def historical_var_cvar(losses, alpha):
    losses = pd.Series(losses).dropna()
    var = losses.quantile(alpha)
    tail = losses[losses >= var]
    return float(var), float(tail.mean()) if len(tail) else np.nan

def compute_score_series(relative_returns, metric_name, config):
    if metric_name == "mean":
        return relative_returns.mean()
    if metric_name == "sharpe":
        return relative_returns.apply(lambda x: sharpe_ratio(x, config.annualisation), axis=0)
    if metric_name == "sortino":
        return relative_returns.apply(lambda x: sortino_ratio(x, config.annualisation), axis=0)
    if metric_name == "calmar":
        return relative_returns.apply(lambda x: calmar_ratio(x, config.annualisation), axis=0)
    raise ValueError(f"Unknown metric: {metric_name}")

def performance_table(strategy_returns, benchmark, relative_returns, config):
    rows = []
    for strategy_id in strategy_returns.columns:
        r = strategy_returns[strategy_id]
        d = relative_returns[strategy_id]
        var, cvar = historical_var_cvar(-r, config.cvar_alpha)
        rows.append({
            "strategy_id": strategy_id,
            "mean_relative_daily": d.mean(),
            "relative_ann_return": d.mean() * config.annualisation,
            "relative_sharpe": sharpe_ratio(d, config.annualisation),
            "strategy_ann_return": annualised_return(r, config.annualisation),
            "strategy_ann_vol": annualised_vol(r, config.annualisation),
            "strategy_sharpe": sharpe_ratio(r, config.annualisation),
            "strategy_sortino": sortino_ratio(r, config.annualisation),
            "strategy_calmar": calmar_ratio(r, config.annualisation),
            "max_drawdown": max_drawdown_from_returns(r),
            "VaR": var,
            "CVaR": cvar,
            "skew": float(r.skew()),
            "excess_kurtosis": float(r.kurtosis()),
        })
    return pd.DataFrame(rows)

perf = performance_table(strategy_returns, benchmark, relative_returns, config).merge(strategy_metadata, on="strategy_id", how="left")
perf.sort_values("relative_sharpe", ascending=False).head(15)

## 9. Bootstrap index generators

We implement three bootstrap schemes:

### IID bootstrap

Samples individual days independently.

### Moving block bootstrap

Samples contiguous blocks of fixed length.

### Stationary bootstrap

Starts a new block with probability $p$, otherwise continues the previous block.

Stationary bootstrap preserves dependence while allowing random block lengths.

In [ ]:
def iid_bootstrap_indices(n, rng):
    return rng.integers(0, n, size=n)

def moving_block_bootstrap_indices(n, block_length, rng):
    indices = []
    while len(indices) < n:
        start = rng.integers(0, n)
        block = [(start + j) % n for j in range(block_length)]
        indices.extend(block)
    return np.array(indices[:n])

def stationary_bootstrap_indices(n, p, rng):
    indices = np.empty(n, dtype=int)
    indices[0] = rng.integers(0, n)
    for t in range(1, n):
        if rng.uniform() < p:
            indices[t] = rng.integers(0, n)
        else:
            indices[t] = (indices[t - 1] + 1) % n
    return indices

# Quick sanity check.
rng_check = np.random.default_rng(config.seed)
pd.DataFrame({
    "iid": iid_bootstrap_indices(10, rng_check),
    "moving_block": moving_block_bootstrap_indices(10, 3, rng_check),
    "stationary": stationary_bootstrap_indices(10, 0.2, rng_check),
})

## 10. White Reality Check implementation

Observed statistic:

$$
T_{obs} = \max_m \sqrt{n}\bar d_m
$$

Bootstrap statistic under null:

$$
T_b^* = \max_m \sqrt{n}\bar{\tilde d}^{*(b)}_m
$$

where:

$$
\tilde d_{m,t} = d_{m,t} - \bar d_m
$$

p-value:

$$
p = \frac{1+\sum_b I(T_b^* \ge T_{obs})}{B+1}
$$

The plus-one adjustment avoids zero p-values.

In [ ]:
def white_reality_check(
    relative_returns,
    config,
    bootstrap_method="stationary",
    metric_name="mean",
    studentized=False,
    spa_filter=False,
):
    rng = np.random.default_rng(config.seed + abs(hash((bootstrap_method, metric_name, studentized, spa_filter))) % 100_000)

    X = relative_returns.copy().replace([np.inf, -np.inf], np.nan).dropna(how="any")
    n = len(X)
    m = X.shape[1]

    if metric_name != "mean":
        observed_scores = compute_score_series(X, metric_name, config)
        selected_strategy = observed_scores.idxmax()
        observed_stat = observed_scores.max()
    else:
        mean_diff = X.mean(axis=0)
        selected_strategy = mean_diff.idxmax()
        if studentized:
            se = X.std(axis=0, ddof=1).replace(0, np.nan) / np.sqrt(n)
            observed_stats_by_strategy = mean_diff / (se + config.studentize_epsilon)
            observed_stat = observed_stats_by_strategy.max()
        else:
            observed_stats_by_strategy = np.sqrt(n) * mean_diff
            observed_stat = observed_stats_by_strategy.max()

    centered = X - X.mean(axis=0)

    if spa_filter and metric_name == "mean":
        # Simplified SPA-style filtering: remove strategies that are clearly poor in sample.
        mean_diff = X.mean(axis=0)
        sd = X.std(axis=0, ddof=1).replace(0, np.nan)
        t_like = np.sqrt(n) * mean_diff / (sd + config.studentize_epsilon)
        keep_cols = t_like[t_like > config.spa_exclusion_threshold].index.tolist()
        if len(keep_cols) >= 2:
            centered = centered[keep_cols]
            X_for_observed = X[keep_cols]
            mean_diff = X_for_observed.mean(axis=0)
            if studentized:
                se = X_for_observed.std(axis=0, ddof=1).replace(0, np.nan) / np.sqrt(n)
                observed_stats_by_strategy = mean_diff / (se + config.studentize_epsilon)
                observed_stat = observed_stats_by_strategy.max()
            else:
                observed_stats_by_strategy = np.sqrt(n) * mean_diff
                observed_stat = observed_stats_by_strategy.max()
            selected_strategy = observed_stats_by_strategy.idxmax()

    boot_stats = np.empty(config.bootstrap_samples)

    for b in range(config.bootstrap_samples):
        if bootstrap_method == "iid":
            idx = iid_bootstrap_indices(n, rng)
        elif bootstrap_method == "moving_block":
            idx = moving_block_bootstrap_indices(n, config.block_length, rng)
        elif bootstrap_method == "stationary":
            idx = stationary_bootstrap_indices(n, config.stationary_bootstrap_p, rng)
        else:
            raise ValueError("Unknown bootstrap method")

        sample = centered.iloc[idx]

        if metric_name == "mean":
            if studentized:
                sample_mean = sample.mean(axis=0)
                sample_se = sample.std(axis=0, ddof=1).replace(0, np.nan) / np.sqrt(n)
                stat = (sample_mean / (sample_se + config.studentize_epsilon)).max()
            else:
                stat = (np.sqrt(n) * sample.mean(axis=0)).max()
        else:
            # For non-mean metrics, use centered pseudo returns and recompute the max metric.
            stat = compute_score_series(sample, metric_name, config).max()

        boot_stats[b] = stat

    p_value = (1.0 + np.sum(boot_stats >= observed_stat)) / (config.bootstrap_samples + 1.0)

    result = {
        "bootstrap_method": bootstrap_method,
        "metric_name": metric_name,
        "studentized": studentized,
        "spa_filter": spa_filter,
        "n_obs": n,
        "n_strategies_used": centered.shape[1],
        "selected_strategy": selected_strategy,
        "observed_stat": float(observed_stat),
        "bootstrap_mean_stat": float(np.mean(boot_stats)),
        "bootstrap_p95_stat": float(np.quantile(boot_stats, 0.95)),
        "bootstrap_p99_stat": float(np.quantile(boot_stats, 0.99)),
        "p_value": float(p_value),
        "reject_null_5pct": bool(p_value < config.significance_level),
    }

    boot_df = pd.DataFrame({
        "bootstrap_stat": boot_stats,
        "bootstrap_method": bootstrap_method,
        "metric_name": metric_name,
        "studentized": studentized,
        "spa_filter": spa_filter,
    })

    return result, boot_df

wrc_stationary_result, wrc_stationary_boot = white_reality_check(
    relative_returns,
    config,
    bootstrap_method="stationary",
    metric_name="mean",
    studentized=False,
    spa_filter=False,
)

pd.Series(wrc_stationary_result)

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(wrc_stationary_boot["bootstrap_stat"], bins=50, alpha=0.75)
plt.axvline(wrc_stationary_result["observed_stat"], linestyle="--", label="observed statistic")
plt.title("White Reality Check Bootstrap Null Distribution")
plt.xlabel("Bootstrap max statistic")
plt.ylabel("Count")
plt.legend()
plt.show()

## 11. Compare bootstrap schemes

IID bootstrap can understate uncertainty if returns are dependent.

Block and stationary bootstrap are usually safer for financial time series.

In [ ]:
bootstrap_scheme_rows = []
bootstrap_stat_frames = []

for method in ["iid", "moving_block", "stationary"]:
    res, boot = white_reality_check(
        relative_returns,
        config,
        bootstrap_method=method,
        metric_name="mean",
        studentized=False,
        spa_filter=False,
    )
    bootstrap_scheme_rows.append(res)
    bootstrap_stat_frames.append(boot.assign(test_name=f"WRC_{method}"))

bootstrap_scheme_comparison = pd.DataFrame(bootstrap_scheme_rows)
bootstrap_scheme_boot_stats = pd.concat(bootstrap_stat_frames, ignore_index=True)

bootstrap_scheme_comparison

## 12. Studentized Reality Check

A studentized statistic divides each mean by its estimated standard error:

$$
t_m = \frac{\bar d_m}{\hat{\sigma}_m/\sqrt{n}}
$$

This prevents high-volatility strategies from dominating purely by scale.

In [ ]:
studentized_rows = []
studentized_boot_frames = []

for method in ["iid", "moving_block", "stationary"]:
    res, boot = white_reality_check(
        relative_returns,
        config,
        bootstrap_method=method,
        metric_name="mean",
        studentized=True,
        spa_filter=False,
    )
    studentized_rows.append(res)
    studentized_boot_frames.append(boot.assign(test_name=f"studentized_{method}"))

studentized_comparison = pd.DataFrame(studentized_rows)
studentized_boot_stats = pd.concat(studentized_boot_frames, ignore_index=True)

studentized_comparison

## 13. Simplified SPA-style correction

White's Reality Check can be conservative because poor strategies inflate the bootstrap distribution.

A simplified SPA-style adjustment removes strategies that are clearly poor before computing the max statistic.

This is not a full Hansen SPA implementation, but it demonstrates the idea:

> Do not let many bad models make it harder for genuinely good models to pass.

In [ ]:
spa_rows = []
spa_boot_frames = []

for method in ["iid", "moving_block", "stationary"]:
    res, boot = white_reality_check(
        relative_returns,
        config,
        bootstrap_method=method,
        metric_name="mean",
        studentized=True,
        spa_filter=True,
    )
    spa_rows.append(res)
    spa_boot_frames.append(boot.assign(test_name=f"spa_style_{method}"))

spa_comparison = pd.DataFrame(spa_rows)
spa_boot_stats = pd.concat(spa_boot_frames, ignore_index=True)

spa_comparison

## 14. Reality Check by performance metric

The Reality Check can use different performance definitions.

We compare:

- mean excess return;
- Sharpe;
- Sortino;
- Calmar.

Mean excess return has the cleanest theory here. The others are included as practical diagnostics.

In [ ]:
metric_rows = []
metric_boot_frames = []

for metric_name in ["mean", "sharpe", "sortino", "calmar"]:
    res, boot = white_reality_check(
        relative_returns,
        config,
        bootstrap_method="stationary",
        metric_name=metric_name,
        studentized=False,
        spa_filter=False,
    )
    metric_rows.append(res)
    metric_boot_frames.append(boot.assign(test_name=f"metric_{metric_name}"))

metric_comparison = pd.DataFrame(metric_rows)
metric_boot_stats = pd.concat(metric_boot_frames, ignore_index=True)

metric_comparison

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(metric_comparison["metric_name"], metric_comparison["p_value"])
plt.axvline(config.significance_level, linestyle="--", label="5% level")
plt.title("Reality Check p-value by Metric")
plt.xlabel("p-value")
plt.ylabel("Metric")
plt.legend()
plt.show()

## 15. Confidence interval for best strategy outperformance

We bootstrap the selected strategy's benchmark-relative mean.

This is not a multiple-testing corrected test by itself.

It is a useful descriptive interval.

In [ ]:
selected_strategy = wrc_stationary_result["selected_strategy"]
selected_diff = relative_returns[selected_strategy].dropna()

def bootstrap_mean_ci(series, config, method="stationary"):
    rng = np.random.default_rng(config.seed + 777)
    x = pd.Series(series).dropna()
    n = len(x)
    means = np.empty(config.bootstrap_samples)

    for b in range(config.bootstrap_samples):
        if method == "iid":
            idx = iid_bootstrap_indices(n, rng)
        elif method == "moving_block":
            idx = moving_block_bootstrap_indices(n, config.block_length, rng)
        else:
            idx = stationary_bootstrap_indices(n, config.stationary_bootstrap_p, rng)

        means[b] = x.iloc[idx].mean()

    return {
        "mean_daily": x.mean(),
        "mean_ann": x.mean() * config.annualisation,
        "ci05_ann": np.quantile(means * config.annualisation, 0.05),
        "ci50_ann": np.quantile(means * config.annualisation, 0.50),
        "ci95_ann": np.quantile(means * config.annualisation, 0.95),
    }, means

selected_ci, selected_boot_means = bootstrap_mean_ci(selected_diff, config, method="stationary")

selected_ci_report = pd.DataFrame([{
    "selected_strategy": selected_strategy,
    **selected_ci,
}])

selected_ci_report

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(selected_boot_means * config.annualisation, bins=50)
plt.axvline(selected_diff.mean() * config.annualisation, linestyle="--", label="observed annualised mean")
plt.axvline(0, linestyle=":", label="zero")
plt.title(f"Bootstrap CI for Selected Strategy Relative Mean: {selected_strategy}")
plt.xlabel("Annualised relative mean")
plt.ylabel("Count")
plt.legend()
plt.show()

## 16. Effect of poor strategies on Reality Check

White's Reality Check can be conservative when many poor strategies are included.

We compare the full library to a filtered library that removes clearly poor strategies based on full-sample relative mean.

This section is diagnostic only. Filtering on full sample is not a production selection method.

In [ ]:
relative_mean = relative_returns.mean(axis=0)
filtered_cols = relative_mean[relative_mean > relative_mean.quantile(0.50)].index.tolist()
relative_returns_filtered = relative_returns[filtered_cols]

filtered_result, filtered_boot = white_reality_check(
    relative_returns_filtered,
    config,
    bootstrap_method="stationary",
    metric_name="mean",
    studentized=False,
    spa_filter=False,
)

poor_strategy_effect = pd.DataFrame([
    {**wrc_stationary_result, "library": "full_library"},
    {**filtered_result, "library": "filtered_top_half_by_relative_mean_diagnostic"},
])

poor_strategy_effect[["library", "n_strategies_used", "selected_strategy", "observed_stat", "bootstrap_p95_stat", "p_value", "reject_null_5pct"]]

## 17. Block length sensitivity

Reality Check p-values can depend on block length.

Short blocks may understate dependence.

Very long blocks may reduce bootstrap diversity.

In [ ]:
block_lengths = [5, 10, 20, 40, 60]
block_sensitivity_rows = []

for block_len in block_lengths:
    cfg = WhiteRealityCheckConfig(
        block_length=block_len,
        stationary_bootstrap_p=1 / block_len,
        bootstrap_samples=1_000,
    )
    res_moving, _ = white_reality_check(
        relative_returns,
        cfg,
        bootstrap_method="moving_block",
        metric_name="mean",
        studentized=False,
        spa_filter=False,
    )
    res_stationary, _ = white_reality_check(
        relative_returns,
        cfg,
        bootstrap_method="stationary",
        metric_name="mean",
        studentized=False,
        spa_filter=False,
    )
    block_sensitivity_rows.append({**res_moving, "block_length": block_len})
    block_sensitivity_rows.append({**res_stationary, "block_length": block_len})

block_sensitivity = pd.DataFrame(block_sensitivity_rows)

block_sensitivity[["bootstrap_method", "block_length", "p_value", "observed_stat", "bootstrap_p95_stat", "reject_null_5pct"]]

In [ ]:
plt.figure(figsize=(10, 6))
for method, sub in block_sensitivity.groupby("bootstrap_method"):
    plt.plot(sub["block_length"], sub["p_value"], marker="o", label=method)
plt.axhline(config.significance_level, linestyle="--", label="5% level")
plt.title("Reality Check p-value Sensitivity to Block Length")
plt.xlabel("Block length")
plt.ylabel("p-value")
plt.legend()
plt.show()

## 18. Placebo all-noise library

Under a pure-noise library, Reality Check should usually not reject.

This is a sanity check on the test procedure.

In [ ]:
noise_returns, noise_benchmark, noise_metadata, noise_regime = simulate_strategy_library(
    config,
    scenario_name="all_noise_placebo",
    all_noise=True,
)

noise_relative = noise_returns.sub(noise_benchmark, axis=0)

noise_wrc_result, noise_wrc_boot = white_reality_check(
    noise_relative,
    config,
    bootstrap_method="stationary",
    metric_name="mean",
    studentized=False,
    spa_filter=False,
)

pd.Series(noise_wrc_result)

## 19. Stronger-alpha library

Under a stronger true-alpha library, Reality Check should be more likely to reject.

In [ ]:
strong_returns, strong_benchmark, strong_metadata, strong_regime = simulate_strategy_library(
    config,
    scenario_name="stronger_true_alpha",
    true_alpha_scale=3.0,
    all_noise=False,
)

strong_relative = strong_returns.sub(strong_benchmark, axis=0)

strong_wrc_result, strong_wrc_boot = white_reality_check(
    strong_relative,
    config,
    bootstrap_method="stationary",
    metric_name="mean",
    studentized=False,
    spa_filter=False,
)

scenario_comparison = pd.DataFrame([
    {**wrc_stationary_result, "scenario": "mixed_true_alpha"},
    {**noise_wrc_result, "scenario": "all_noise_placebo"},
    {**strong_wrc_result, "scenario": "stronger_true_alpha"},
])

scenario_comparison[["scenario", "selected_strategy", "observed_stat", "bootstrap_p95_stat", "p_value", "reject_null_5pct"]]

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(scenario_comparison["scenario"], scenario_comparison["p_value"])
plt.axvline(config.significance_level, linestyle="--", label="5% level")
plt.title("White Reality Check p-value by Scenario")
plt.xlabel("p-value")
plt.ylabel("Scenario")
plt.legend()
plt.show()

## 20. Reality-check adjusted report

We produce a report for the selected strategy:

- naive performance;
- benchmark-relative performance;
- Reality Check p-value;
- SPA-style p-value;
- bootstrap confidence interval;
- whether null is rejected.

In [ ]:
selected_perf = perf[perf["strategy_id"] == selected_strategy].iloc[0].to_dict()
selected_meta = strategy_metadata[strategy_metadata["strategy_id"] == selected_strategy].iloc[0].to_dict()

spa_stationary = spa_comparison[spa_comparison["bootstrap_method"] == "stationary"].iloc[0].to_dict()
studentized_stationary = studentized_comparison[studentized_comparison["bootstrap_method"] == "stationary"].iloc[0].to_dict()

reality_check_report = pd.DataFrame([{
    "selected_strategy": selected_strategy,
    "has_true_alpha_in_simulation": bool(selected_meta["has_true_alpha"]),
    "true_alpha_ann": selected_meta["true_alpha_ann"],
    "strategy_ann_return": selected_perf["strategy_ann_return"],
    "strategy_sharpe": selected_perf["strategy_sharpe"],
    "relative_ann_return": selected_perf["relative_ann_return"],
    "relative_sharpe": selected_perf["relative_sharpe"],
    "max_drawdown": selected_perf["max_drawdown"],
    "CVaR": selected_perf["CVaR"],
    "white_reality_check_p_value": wrc_stationary_result["p_value"],
    "white_reality_check_reject_5pct": wrc_stationary_result["reject_null_5pct"],
    "studentized_wrc_p_value": studentized_stationary["p_value"],
    "spa_style_p_value": spa_stationary["p_value"],
    "bootstrap_ci05_ann_relative_mean": selected_ci["ci05_ann"],
    "bootstrap_ci95_ann_relative_mean": selected_ci["ci95_ann"],
}])

reality_check_report

## 21. Governance flags

A strategy library should be reviewed if:

1. WRC p-value is not significant;
2. SPA-style result disagrees materially with WRC;
3. p-value is sensitive to block length;
4. all-noise placebo rejects too often;
5. stronger-alpha scenario still fails to reject;
6. selected strategy's confidence interval includes zero;
7. selected strategy has poor drawdown or CVaR.

In [ ]:
stationary_p = wrc_stationary_result["p_value"]
spa_p = spa_stationary["p_value"]
block_p_range = block_sensitivity["p_value"].max() - block_sensitivity["p_value"].min()
ci_includes_zero = selected_ci["ci05_ann"] <= 0 <= selected_ci["ci95_ann"]

governance_flags = pd.DataFrame([{
    "selected_strategy": selected_strategy,
    "wrc_stationary_p_value": stationary_p,
    "studentized_stationary_p_value": studentized_stationary["p_value"],
    "spa_style_stationary_p_value": spa_p,
    "block_p_value_range": block_p_range,
    "noise_placebo_p_value": noise_wrc_result["p_value"],
    "strong_alpha_p_value": strong_wrc_result["p_value"],
    "selected_ci_includes_zero": ci_includes_zero,
    "selected_max_drawdown": selected_perf["max_drawdown"],
    "selected_CVaR": selected_perf["CVaR"],
    "flag_wrc_not_significant": bool(stationary_p >= config.significance_level),
    "flag_studentized_not_significant": bool(studentized_stationary["p_value"] >= config.significance_level),
    "flag_spa_disagrees_with_wrc": bool((spa_p < config.significance_level) != (stationary_p < config.significance_level)),
    "flag_block_sensitivity_high": bool(block_p_range > 0.20),
    "flag_noise_placebo_rejects": bool(noise_wrc_result["p_value"] < config.significance_level),
    "flag_strong_alpha_fails_to_reject": bool(strong_wrc_result["p_value"] >= config.significance_level),
    "flag_selected_ci_includes_zero": bool(ci_includes_zero),
    "flag_drawdown_below_minus_25pct": bool(selected_perf["max_drawdown"] < -0.25),
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_wrc_not_significant",
        "flag_studentized_not_significant",
        "flag_spa_disagrees_with_wrc",
        "flag_block_sensitivity_high",
        "flag_noise_placebo_rejects",
        "flag_strong_alpha_fails_to_reject",
        "flag_selected_ci_includes_zero",
        "flag_drawdown_below_minus_25pct",
    ]
].any(axis=1)

governance_flags

## 22. Audit checks

Numerical and process checks:

1. relative returns equal strategy minus benchmark;
2. bootstrap p-values are in $[0,1]$;
3. bootstrap statistics are finite;
4. selected strategy exists;
5. centered returns have approximately zero mean;
6. scenario tests completed;
7. block sensitivity has all requested block lengths.

In [ ]:
audit_rows = []

relative_check = strategy_returns.sub(benchmark, axis=0)
max_rel_diff = float((relative_check - relative_returns).abs().max().max())
audit_rows.append({
    "check": "relative_returns_equal_strategy_minus_benchmark",
    "value": max_rel_diff,
    "passed": bool(max_rel_diff < 1e-12),
})

all_p_values = pd.concat([
    bootstrap_scheme_comparison["p_value"],
    studentized_comparison["p_value"],
    spa_comparison["p_value"],
    metric_comparison["p_value"],
    block_sensitivity["p_value"],
    scenario_comparison["p_value"],
])
p_values_valid = bool(((all_p_values >= 0) & (all_p_values <= 1)).all())
audit_rows.append({
    "check": "all_p_values_in_unit_interval",
    "value": p_values_valid,
    "passed": p_values_valid,
})

boot_stats_finite = bool(np.isfinite(bootstrap_scheme_boot_stats["bootstrap_stat"]).all())
audit_rows.append({
    "check": "bootstrap_statistics_finite",
    "value": boot_stats_finite,
    "passed": boot_stats_finite,
})

selected_exists = selected_strategy in strategy_returns.columns
audit_rows.append({
    "check": "selected_strategy_exists",
    "value": selected_exists,
    "passed": selected_exists,
})

centered = relative_returns - relative_returns.mean(axis=0)
max_centered_mean = float(centered.mean(axis=0).abs().max())
audit_rows.append({
    "check": "centered_relative_returns_zero_mean",
    "value": max_centered_mean,
    "passed": bool(max_centered_mean < 1e-12),
})

scenario_complete = bool(len(noise_wrc_boot) > 0 and len(strong_wrc_boot) > 0)
audit_rows.append({
    "check": "placebo_and_strong_scenarios_completed",
    "value": scenario_complete,
    "passed": scenario_complete,
})

block_lengths_complete = bool(set(block_sensitivity["block_length"]) == set(block_lengths))
audit_rows.append({
    "check": "block_sensitivity_all_lengths_present",
    "value": block_lengths_complete,
    "passed": block_lengths_complete,
})

audit_report = pd.DataFrame(audit_rows)

audit_report

## 23. Practical checklist for White Reality Check

Before claiming a strategy is superior:

1. **Define the strategy universe**  
   Include all tested variants, not only survivors.

2. **Choose benchmark-relative performance**  
   Reality Check tests outperformance versus a benchmark.

3. **Use dependence-aware bootstrap**  
   Prefer block or stationary bootstrap over IID for financial returns.

4. **Center under the null**  
   Bootstrap centered performance differences.

5. **Use the max statistic**  
   The test must account for the best of many strategies.

6. **Report p-values, not just Sharpe**  
   The best Sharpe can still be insignificant.

7. **Check block-length sensitivity**  
   Bootstrap settings should not decide the conclusion.

8. **Run placebo controls**  
   All-noise libraries should not often reject.

9. **Consider SPA-style adjustment**  
   WRC can be conservative when many bad models are included.

10. **Save the full research universe**  
   Hidden failed strategies invalidate data-snooping correction.

## 24. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed/white_reality_check_bootstrap_v1")
output_dir.mkdir(parents=True, exist_ok=True)

paths = {
    "strategy_returns": output_dir / "strategy_returns.csv",
    "benchmark": output_dir / "benchmark.csv",
    "relative_returns": output_dir / "relative_returns.csv",
    "strategy_metadata": output_dir / "strategy_metadata.csv",
    "regime_info": output_dir / "regime_info.csv",
    "performance": output_dir / "performance.csv",
    "wrc_stationary_result": output_dir / "wrc_stationary_result.csv",
    "wrc_stationary_bootstrap": output_dir / "wrc_stationary_bootstrap.csv",
    "bootstrap_scheme_comparison": output_dir / "bootstrap_scheme_comparison.csv",
    "bootstrap_scheme_boot_stats": output_dir / "bootstrap_scheme_boot_stats.csv",
    "studentized_comparison": output_dir / "studentized_comparison.csv",
    "studentized_boot_stats": output_dir / "studentized_boot_stats.csv",
    "spa_comparison": output_dir / "spa_comparison.csv",
    "spa_boot_stats": output_dir / "spa_boot_stats.csv",
    "metric_comparison": output_dir / "metric_comparison.csv",
    "metric_boot_stats": output_dir / "metric_boot_stats.csv",
    "selected_ci_report": output_dir / "selected_ci_report.csv",
    "poor_strategy_effect": output_dir / "poor_strategy_effect.csv",
    "block_sensitivity": output_dir / "block_sensitivity.csv",
    "noise_wrc_result": output_dir / "noise_wrc_result.csv",
    "strong_wrc_result": output_dir / "strong_wrc_result.csv",
    "scenario_comparison": output_dir / "scenario_comparison.csv",
    "reality_check_report": output_dir / "reality_check_report.csv",
    "governance_flags": output_dir / "governance_flags.csv",
    "audit_report": output_dir / "audit_report.csv",
    "manifest": output_dir / "manifest.json",
}

strategy_returns.to_csv(paths["strategy_returns"])
benchmark.to_frame("benchmark_return").to_csv(paths["benchmark"])
relative_returns.to_csv(paths["relative_returns"])
strategy_metadata.to_csv(paths["strategy_metadata"], index=False)
regime_info.to_csv(paths["regime_info"])
perf.to_csv(paths["performance"], index=False)
pd.Series(wrc_stationary_result).to_frame("value").to_csv(paths["wrc_stationary_result"])
wrc_stationary_boot.to_csv(paths["wrc_stationary_bootstrap"], index=False)
bootstrap_scheme_comparison.to_csv(paths["bootstrap_scheme_comparison"], index=False)
bootstrap_scheme_boot_stats.to_csv(paths["bootstrap_scheme_boot_stats"], index=False)
studentized_comparison.to_csv(paths["studentized_comparison"], index=False)
studentized_boot_stats.to_csv(paths["studentized_boot_stats"], index=False)
spa_comparison.to_csv(paths["spa_comparison"], index=False)
spa_boot_stats.to_csv(paths["spa_boot_stats"], index=False)
metric_comparison.to_csv(paths["metric_comparison"], index=False)
metric_boot_stats.to_csv(paths["metric_boot_stats"], index=False)
selected_ci_report.to_csv(paths["selected_ci_report"], index=False)
poor_strategy_effect.to_csv(paths["poor_strategy_effect"], index=False)
block_sensitivity.to_csv(paths["block_sensitivity"], index=False)
pd.Series(noise_wrc_result).to_frame("value").to_csv(paths["noise_wrc_result"])
pd.Series(strong_wrc_result).to_frame("value").to_csv(paths["strong_wrc_result"])
scenario_comparison.to_csv(paths["scenario_comparison"], index=False)
reality_check_report.to_csv(paths["reality_check_report"], index=False)
governance_flags.to_csv(paths["governance_flags"], index=False)
audit_report.to_csv(paths["audit_report"], index=False)

manifest = {
    "dataset_name": "white_reality_check_bootstrap_outputs",
    "schema_version": "white_reality_check_bootstrap_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "selected_strategy": selected_strategy,
    "wrc_stationary_result": wrc_stationary_result,
    "studentized_stationary_result": studentized_stationary,
    "spa_stationary_result": spa_stationary,
    "scenario_comparison": scenario_comparison.to_dict(orient="records"),
    "reality_check_report": reality_check_report.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "notes": [
        "White Reality Check tests whether the best strategy outperforms benchmark after accounting for data snooping.",
        "The core statistic is the maximum centered bootstrap performance differential.",
        "IID, moving block and stationary bootstrap schemes are compared.",
        "A studentized version and simplified SPA-style filter are included as practical diagnostics.",
        "All-noise and stronger-alpha scenarios are included as controls.",
        "Block-length sensitivity is reported.",
        "This notebook focuses on research significance after multiple strategy trials, not production readiness."
    ],
}

paths["manifest"].write_text(json.dumps(manifest, indent=2, default=str))

paths["reality_check_report"], paths["bootstrap_scheme_comparison"], paths["scenario_comparison"], paths["governance_flags"], paths["manifest"]

## 25. Limitations

### Simplified SPA

The SPA-style filter is educational and simplified. A full Hansen SPA implementation is more nuanced.

### Synthetic data

The strategy library is simulated. Real strategy libraries have correlated parameter variants, hidden failed trials, changing costs, and survivorship issues.

### Bootstrap assumptions

Block and stationary bootstrap preserve some dependence, but they do not perfectly model all financial time-series structure.

### Strategy universe definition

Reality Check is only as honest as the universe of strategies included. Omitting failed trials understates data-snooping risk.

### Metric dependence

Results vary by performance metric. Mean excess return has cleaner interpretation than nonlinear metrics like Calmar.

### No purged labels here

This notebook works with strategy returns. If strategies are ML models trained on overlapping labels, purged CV must be handled upstream.

## 26. Research frontier and extensions

Important extensions include:

1. full Hansen SPA implementation;
2. Model Confidence Set procedures;
3. White Reality Check with correlated strategy clusters;
4. Reality Check after purged cross-validation;
5. transaction-cost uncertainty bootstrap;
6. regime-conditioned Reality Check;
7. block-length selection methods;
8. bootstrap for path-dependent metrics;
9. Reality Check for portfolio allocation methods;
10. experiment registry integration to ensure all tried strategies are included.

## 27. Suggested follow-up notebooks

This notebook naturally leads to:

1. **05_10_research_audit_trail_and_manifest**  
   Record every strategy variant needed for honest Reality Check.

2. **05_11_live_shadow_mode_monitoring**  
   Compare Reality Check conclusions to live paper performance.

3. **05_12_full_research_to_production_checklist**  
   Combine WRC, PBO, DSR, costs and governance.

4. **07_06_research_governance_capstone**  
   Full alpha-research validation framework.

## 28. Summary

This notebook implemented White's Reality Check with bootstrap.

It showed how to:

1. simulate a strategy library and benchmark;
2. compute benchmark-relative performance;
3. implement IID bootstrap;
4. implement moving block bootstrap;
5. implement stationary bootstrap;
6. center performance under the null;
7. compute the max-performance Reality Check statistic;
8. estimate White Reality Check p-values;
9. compare bootstrap schemes;
10. compute studentized Reality Check;
11. demonstrate simplified SPA-style filtering;
12. test multiple performance metrics;
13. compute bootstrap confidence intervals;
14. study the effect of poor strategies;
15. test block-length sensitivity;
16. run all-noise and stronger-alpha controls;
17. create a reality-check adjusted strategy report;
18. create governance flags and audit checks;
19. save all outputs and manifests.

The key computational takeaway:

> White's Reality Check bootstraps the maximum performance statistic across all tried strategies under the null of no superior model.

The key financial takeaway:

> The best strategy in a large research library is not significant just because it is the best; it must beat the best that data snooping can manufacture.

## 29. Further reading

- White, "A Reality Check for Data Snooping."
- Hansen, "A Test for Superior Predictive Ability."
- Hansen, Lunde and Nason on Model Confidence Sets.
- López de Prado, *Advances in Financial Machine Learning.*
- Bailey et al. on backtest overfitting.
- Harvey, Liu and Zhu on multiple testing in factor research.
- Institutional research-governance literature on experiment registries and model validation.